In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_data=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_data=pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
women=train_data.loc[train_data.Sex=='female']['Survived']
men=train_data.loc[train_data.Sex=='male']['Survived']
rate_women = sum(women) / len(women)
rate_men=sum(men)/len(men)
print(rate_women)
print(rate_men)

In [ ]:
for pclass in [1,2,3]:
    assume=train_data.loc[(train_data.Sex=='female') & (train_data.Pclass==pclass)]['Survived'] 
    print(sum(assume)/len(assume))

In [ ]:
train_data.isna().sum()

In [ ]:
work = train_data.copy()
work['Sex']=work['Sex'].map({'male':0,'female':1})
work[ "Embarked" ] = work[ "Embarked" ].fillna( "S" )
work[ "Embarked" ] = work[ "Embarked" ]. map ({ "S" : 0 , "C" : 1 , "Q" : 2 })
work[ "Age" ] = work[ "Age" ].fillna(work[ "Age" ].median())
print (work[[ "Sex" , "Embarked" , "Age" ]].head())

In [ ]:
features = [ "Pclass" , "Sex" , "Age" , "SibSp" , "Parch" , "Fare" , "Embarked" ]
X=work[features]
y=work['Survived']
print(X.head())
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

print("number of people in the training set‌:", len(train_X))
print("number of people in the testing set‌:", len(val_X))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=1)
model.fit(train_X, train_y)

print("Model training is complete‌")

In [ ]:
from sklearn.metrics import accuracy_score

preds = model.predict(val_X)

print("test_score:", accuracy_score(val_y, preds))

In [ ]:
importance = pd.Series(model.feature_importances_, index=features)
print(importance.sort_values(ascending=False))

In [ ]:
for max_leaf_nodes in [5, 50, 500, 5000]:
    model = RandomForestClassifier(max_leaf_nodes=max_leaf_nodes, random_state=1)
    model.fit(train_X, train_y)
    preds = model.predict(val_X)
    score = accuracy_score(val_y, preds)
    print("Max leaf nodes: %d \t Accuracy: %g" % (max_leaf_nodes, score))

In [ ]:
work_test = test_data.copy()

work_test["Sex"] = work_test["Sex"].map({"male": 0, "female": 1})
work_test["Embarked"] = work_test["Embarked"].fillna("S")
work_test["Embarked"] = work_test["Embarked"].map({"S": 0, "C": 1, "Q": 2})
work_test["Age"] = work_test["Age"].fillna(work["Age"].median())
work_test["Fare"] = work_test["Fare"].fillna(work["Fare"].median())

X_test = work_test[features]
print(X_test.head())
print(X_test.shape)

In [ ]:
final_model = RandomForestClassifier(max_leaf_nodes=50, random_state=1)
final_model.fit(X, y)

test_preds = final_model.predict(X_test)

output = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": test_preds.astype(int),
})

output.to_csv("submission.csv", index=False)

print(output.head())
print("Total:", len(output))
print("Predicted survived:", test_preds.sum())